# Speech Recognition With Whisper

## Task 1: Load the Libraries

In [1]:
import pandas as pd
import torch
from datasets import load_dataset, load_metric, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainer, Seq2SeqTrainingArguments, pipeline
import evaluate
from huggingface_hub import interpreter_login
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import gradio as gr

/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.0.2) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


## Task 2: Prepare the Enviornment

In [2]:
interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|
    
    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## Task 3: Load the Dataset

In [ ]:
data = load_dataset("EducativeCS2023/dummy_en_asr")
print(data)

In [ ]:
data = data.cast_column("audio", Audio(sampling_rate=16_000))
data['train'][0]

## Task 4: Compute the WER of the Default Model

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language='English', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)

In [ ]:
def map_to_pred(batch):
    audio = batch["audio"]
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
    batch["reference"] = processor.tokenizer._normalize(batch['sentence'])

    with torch.no_grad():
        predicted_ids = model.generate(input_features.to(device), language='English', task='transcribe')[0]
    transcription = processor.decode(predicted_ids)
    batch["prediction"] = processor.tokenizer._normalize(transcription)
    return batch

result = data['test'].map(map_to_pred)

In [ ]:
wer = evaluate.load("wer")
print(100 * wer.compute(references=result["reference"], predictions=result["prediction"]))

# Prepare for Training

## Task 5: Display the Loaded Data

In [ ]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    return pd.DataFrame(dataset[picks])


show_random_elements(data['train'].remove_columns(["audio"])).head()

## Task 6: Prepare the Dataset

In [ ]:
def prepare_dataset(batch):

    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

data['train'] = data['train'].map(prepare_dataset, remove_columns=data.column_names["train"], num_proc=2)
data['validation'] = data['validation'].map(prepare_dataset, remove_columns=data.column_names["validation"], num_proc=2)

# Training and Evaluation

## Task 7: Define a Data Collator

In [ ]:
@dataclass
class DataCollatorSpeech:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
     
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeech(processor=processor)

## Task 8: Define Evaluation Methods

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    value = 100 * wer.compute(predictions=pred_str, references=label_str)

    return {"wer": value}

## Task 9: Define Training Configuration

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
new_path = "./whisper-en-tiny" 

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=new_path,  # change to a repo name of your choice
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=120,
    evaluation_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=60,
    eval_steps=60,
    logging_steps=60,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

## Task 10: Train the Model

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

In [ ]:
processor.save_pretrained(training_args.output_dir)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub()

## Task 11: Evaluate the Model

In [ ]:
result = data['test'].map(map_to_pred)

In [ ]:
print(100 * wer.compute(references=result["reference"], predictions=result["prediction"]))

## Task 12: Deploy With Gradio

In [ ]:
pipe = pipeline("automatic-speech-recognition", model="EducativeCS2023/whisper-en-tiny-trained")

def inference(speech_file):
  return pipe(speech_file)["text"]

gr.Interface(inference,gr.Audio(source="microphone", type="filepath"),"text").launch(share=True) 

# End